# 02 — MRI Preprocessing + Augmentation

1. Loads Baby Open Brains BIDS dataset (10 T1w/T2w subjects)
2. Extracts axial/coronal/sagittal slices → `datasets/processed/mri/` (10 real `.npz`)
3. **Runs `balance_mri.py` to generate 10,000 synthetic augmented samples** → `datasets/mri/augmented/`

**Run order: 01 → 02 → 03 → 04 → 05 → 06**

In [ ]:
# CELL 1: Clone / pull repo
import os
if not os.path.exists('/content/earlyMind'):
    !git clone https://github.com/Rickykapoor/earlyMind.git /content/earlyMind
%cd /content/earlyMind
!git pull origin main

In [ ]:
# CELL 2: Install dependencies
!pip install -q mne nibabel nilearn openneuro-py torch torchvision \
  streamlit plotly scikit-learn pytorch-tabnet \
  xgboost catboost tqdm pyyaml scipy

In [ ]:
# CELL 3: Check GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# CELL 4: Download raw MRI dataset from GitHub Releases
!mkdir -p datasets/mri/baby_open_brains
!wget -qO mri_raw.tar.gz https://github.com/Rickykapoor/earlyMind/releases/download/v1.0.0-data/mri_raw.tar.gz
!tar -xzf mri_raw.tar.gz -C datasets/mri
print('Raw MRI dataset ready.')

In [ ]:
# CELL 5: Setup paths + inspect participants
import sys
sys.path.insert(0, '/content/earlyMind')

from src.config import cfg
from src.data.mri_loader import process_mri_dataset
import pandas as pd
import numpy as np

print('MRI raw dir: ', cfg.paths.mri_raw)
print('MRI proc dir:', cfg.paths.mri_processed)

tsv_path = cfg.paths.mri_raw / 'participants.tsv'
if tsv_path.exists():
    df_part = pd.read_csv(tsv_path, sep='\t')
    print('Participants TSV columns:', df_part.columns.tolist())
    display(df_part)
else:
    print('participants.tsv not found')

In [ ]:
# CELL 6: Run MRI preprocessing (10 real subjects → datasets/processed/mri/)
results = process_mri_dataset(
    mri_dir=cfg.paths.mri_raw,
    output_dir=cfg.paths.mri_processed,
)

print('\n=== Preprocessing Results ===')
for pid, info in results.items():
    print(f'  {pid}: slices={info["slices"].shape}, age={info["age_months"]:.1f}mo, '
          f'label={info["label"]}, DQ={info["dq"]:.1f}')

import glob
real_files = glob.glob('datasets/processed/mri/*.npz')
print(f'\n  Real .npz files written: {len(real_files)}')

In [ ]:
# CELL 7: Visualize slices for first subject
import matplotlib.pyplot as plt

first_pid = list(results.keys())[0]
slices = results[first_pid]['slices']  # (3, 64, 64)
titles = ['Axial', 'Coronal', 'Sagittal']

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i, (ax, title) in enumerate(zip(axes, titles)):
    ax.imshow(slices[i], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'{first_pid} — {title}', fontsize=12)
    ax.axis('off')
plt.tight_layout()
cfg.paths.reports.mkdir(parents=True, exist_ok=True)
plt.savefig(cfg.paths.reports / 'mri_slices.png', dpi=100)
plt.show()
print(f'Slice shape: {slices.shape}, range: [{slices.min():.3f}, {slices.max():.3f}]')

In [ ]:
# CELL 8: *** RUN AUGMENTATION ***
# Generates 10,000 synthetic MRI samples from the 10 real subjects.
# Uses seed=42 for full reproducibility (same result as the original Mac run on Apr 12).
# Output: datasets/mri/augmented/  (~420 MB, ~10,000 .npz files)
# Runtime: ~5-15 minutes on Colab CPU (augmentation is CPU-bound)
print('Running MRI augmentation pipeline...')
print('Target: 10,000 synthetic samples | Seed: 42 | DQ noise std: 5.0')
print()

!python scripts/balance_mri.py --skip-preprocess --target 10000 --seed 42 --dq-noise-std 5.0

import glob
aug_files = glob.glob('datasets/mri/augmented/*.npz')
print(f'\n  Augmented files generated: {len(aug_files):,}')

if len(aug_files) < 9000:
    raise RuntimeError(f'Expected ~10,000 augmented files but got {len(aug_files)}. Check the script output above.')

print('  Augmentation complete! Proceeding to notebook 03 or 04.')

In [ ]:
# CELL 9: Verify augmented class distribution
import numpy as np
import collections

labels = []
for f in sorted(aug_files)[:500]:   # sample first 500 for speed
    d = np.load(f)
    labels.append(int(d['label']))

dist = collections.Counter(labels)
label_names = {0: 'Typical', 1: 'Borderline', 2: 'Mild ID', 3: 'Moderate ID', 4: 'Severe ID', 5: 'Profound ID'}
print('=== Augmented Class Distribution (sample of 500) ===')
for k in sorted(dist):
    pct = dist[k] / 500 * 100
    bar = '█' * int(pct / 2)
    print(f'  {label_names.get(k, k):12s}: {dist[k]:5d}  ({pct:.1f}%)  {bar}')

In [ ]:
# CELL 10: Commit and push
!git add reports/ datasets/processed/
!git commit -m 'colab: 02_mri_preprocess + augmentation completed (10,000 synthetic samples)'
!git push origin main